# Fair $k$-means with proportional representation along a continuous sensitive attribute

#### Libraries

In [41]:
from importlib import import_module
from tabulate import tabulate

from src import init, metrics
from src.algorithms import fairkmeans_prc

#### Parameters

In [42]:
DATASET_NAME = 'motor'           # ['adult', 'compas', 'crime', 'motor']
N_CLUSTERS = 4
RANDOM_STATE = 4
INIT_METHOD = 'kmeans_plusplus'
LAMBDA_ = .5
WINDOW_SIZE =3

#### Load dataset

In [43]:
dataset_module = import_module(f"src.datasets.{DATASET_NAME}")
_, X, s, n_nonsensitive = dataset_module.load()

Loading the processed Motor Insurance dataset (motor) from 'data/datasets/motor/motor.csv'
╭───────────┬─────────────┬──────────────┬─────────────────────────────┬───────────────────────╮
│   dataset │   # objects │   # features │   # nonsensitive attributes │   sensitive attribute │
├───────────┼─────────────┼──────────────┼─────────────────────────────┼───────────────────────┤
│     motor │       36311 │          521 │                          11 │                   Age │
╰───────────┴─────────────┴──────────────┴─────────────────────────────┴───────────────────────╯


#### Load initialised centroids

In [44]:
init_centroids = init.load(
    dataset_name=DATASET_NAME, n_clusters=N_CLUSTERS,
    init_method=INIT_METHOD, random_state=RANDOM_STATE
    )

Loading centroids from 'data/init/motor/k=4/kmeans_plusplus/motor.k4.kmeans_plusplus.r4.csv'
  shape: (4, 521)


#### Run the algorithm

In [45]:
import time
start_time = time.perf_counter()

c, centroids, info, stats = fairkmeans_prc.run(
    X=X.copy(), n_nonsensitive=n_nonsensitive, s=s.copy(),
    n_clusters=N_CLUSTERS, init_centroids=init_centroids,
    dataset_name=DATASET_NAME, init_method=INIT_METHOD,
    random_state=RANDOM_STATE, window_size=WINDOW_SIZE,
    lambda_=LAMBDA_
    )
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.4f} seconds")

Initiating fair k-means with proportional representation along a continuous sensitive attribute
╭───────────┬───────────────────────┬──────────────┬─────────────────┬────────────────┬────────────────┬───────────┬───────────────┬────────────╮
│   dataset │   sensitive_attribute │   n_clusters │     init_method │   random_state │      algorithm │   lambda_ │   window_size │   max_iter │
├───────────┼───────────────────────┼──────────────┼─────────────────┼────────────────┼────────────────┼───────────┼───────────────┼────────────┤
│     motor │                   Age │            4 │ kmeans_plusplus │              4 │ fairkmeans_prc │       0.5 │             3 │        100 │
╰───────────┴───────────────────────┴──────────────┴─────────────────┴────────────────┴────────────────┴───────────┴───────────────┴────────────╯
Running algorithm
╭────────┬────────────────┬─────────────────┬─────────────┬─────────────────┬──────────────────┬───────────────────┬──────────────────╮
│   iter │   utility

#### Evaluate

In [46]:
import pandas as pd
with pd.ExcelWriter('./result/motor/LabelFair_K4L5_4.xlsx') as writer:
    c.to_excel(writer, sheet_name='Cluster', index=False)
    centroids.to_excel(writer, sheet_name='Centroid', index=False)
    info.to_excel(writer, sheet_name='Info', index=False)
    stats.to_excel(writer, sheet_name='Stats', index=False)

In [47]:
scores = metrics.evaluate(
    X=X, n_nonsensitive=n_nonsensitive, s=s, c=c, centroids=centroids,
    n_clusters=N_CLUSTERS, window_size=WINDOW_SIZE
    )
print(tabulate(scores.to_frame(), tablefmt='rounded_outline'))

╭─────────────────────────────────────┬─────────────╮
│ k-means objective                   │ 0.473407    │
│ max ks statistic                    │ 0.0018221   │
│ max emd                             │ 0.000455004 │
│ pooling window loss (size=3)        │ 0.00449243  │
│ Modified Abraham (2020)'s deviation │ 0.000162924 │
│ Ziko (2021)'s fairness error        │ 0.000771523 │
│ Bera (2019)'s generalised balance   │ 0.863911    │
╰─────────────────────────────────────┴─────────────╯


In [48]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(4, 3))

# Plot the overall dataset distribution (dashed black)
sns.kdeplot(data=s, fill=False, color="black", linestyle="--", label="dataset")  # data1 as in data 2 we dont have age column. Also it will not affect as it drawing for whole dataset

# Plot each cluster's age distribution
clusters = sorted(c.unique())
palette = sns.color_palette("tab10", len(clusters))

for i, j in enumerate(clusters):
    subset = s[c == j]  # here for which points we will need original dataset or data1
    sns.kdeplot(subset, fill=False, color=palette[i], label=f"cluster_{i} ({len(subset)} objects)")

plt.xlabel("age")
plt.ylabel("proportion")
plt.title("Age Distribution : Clusters vs Full Dataset")
plt.legend()
plt.tight_layout()
#plt.show()
plt.savefig("./result/motor/Plot_LabelFair_K4L5_4.png")
plt.close()